In [ ]:
from pathlib import Path
from metasmith.python_api import Agent, Source, SshSource, Std, DataInstanceLibrary
from metasmith.python_api import DataTypeLibrary, Endpoint, WorkflowTask

dtypes, containers, transforms = Std()

smith = Agent(
    home = SshSource("fir", Path("/scratch/phyberos/metasmith")).AsSource(),
    setup_commands=[
        "module load StdEnv/2023",
        "module load apptainer/1.3.5",
    ]
)
# smith.Deploy()

In [ ]:
# https://training.nextflow.io/2.1.3/side_quests/workflows_of_workflows/#13-make-the-workflow-composable
# use this for array jobs

In [ ]:
lib = DataTypeLibrary()
lib.types = {"a": Endpoint(dtypes["short_reads"].properties|{"test"})}
lib.Save(Path("./cache/dlib_test"))

In [ ]:
inputs = DataInstanceLibrary("./cache/new_sra_accessions.xgdb")
inputs.AddTypeLibrary("acc", lib)

In [ ]:
DataTypeLibrary.Load("./cache/dlib_test")["a"]

In [ ]:
# inputs = DataInstanceLibrary.Load("./cache/flye_inputs.xgdb")
inputs = DataInstanceLibrary.Load("./cache/long_reads.xgdb")
# inputs = DataInstanceLibrary.Load("./cache/asm.xgdb")
for p, n, e in inputs.Iterate():
    print(n, p, e, e.parents)

In [ ]:
for p, n, t in containers.Iterate():
    with open(containers.location/p) as f:
        print(n)

In [ ]:
dtypes.types

In [ ]:
with open("./cache/slurm_account_fir") as f:
    SLURM_ACCOUNT = f.read()

In [ ]:
task = smith.GenerateWorkflow(
    # seed=1,
    # max_refine=1024,
    # max_iter=1024,
    given      = [containers, inputs],
    transforms = [transforms],
    targets    = [dtypes["long_reads_assembly"]],
    # targets    = [dtypes["coding_sequences"]],
    # targets    = [dtypes["functional_annotations"]]
    # targets    = [dtypes["functional_annotations"].WithLineage([dtypes["long_reads"]])],
    # targets    = [dtypes["functional_annotations"].WithLineage([dtypes["hybrid_assembly"]])],
    # targets    = [dtypes["kofamscan_annotations"].WithLineage([dtypes["long_reads"]])],
    config = dict(
        nextflow = dict(
            preset="slurm",
            slurm_account=SLURM_ACCOUNT,
            cpus=8,
        )
    ),
)
for step in task.plan.steps:
    print(step.transform.name)
# task.RenderDAG("./cache/dag")

In [ ]:
WorkflowTask.Merge()

In [ ]:
smith.StageWorkflow(task, on_exist="clear")

In [ ]:
smith.RunWorkflow(task)

In [ ]:
smith.CheckWorkflow(task)

In [ ]:
# asm = DataInstanceLibrary("./cache/asm.xgdb")
# asm.Add(
#     items = [
#         ("/home/tony/workspace/projects/Toluene_resistence_evolution/data/gene_centric/nucleotide/fosmids.spades_meta.gt29kb.fna", "scadc.fna", "std::assembly"),
#     ],
# )
# asm.PruneTypes()

In [ ]:
# fin = DataInstanceLibrary("./cache/flye_inputs.xgdb")
# fin.Add(
#     items = [
#         (Path("./cache/flye_in/miniasm_estimate").absolute(), "miniasm_estimate", "std::miniasm_estimate"),
#         (Path("./cache/flye_in/lr_ss10.fastq").absolute(), "lr_ss10.fastq", "std::long_reads_filtered"),
#     ],
# )
# fin.AddParentsTo("lr_ss10.fastq", [fin.Get("miniasm_estimate")])
# fin.PruneTypes()

In [ ]:
# import os
# r = Path("/home/tony/workspace/tools/Metasmith/src/metasmith/std/containers")
# for f in r.iterdir():
#     os.system(f"""\
#         cd {r}
#         git mv {f.name} {f.name}.oci.uri
#     """)